# AniList Anime Dataset — Data Cleaning Script
> Nova IMS · 2025/2026 · Group 16

In [1]:
"""
Steps:
  1. Load CSV
  2. Drop useless columns
  3. Fix data types
  4. Basic data quality (nulls, duplicates)
  5. Parse JSON columns → Python objects
  6. Extract separate collections (characters, staff, studios, tags, reviews)
  7. Save 6 JSON files → 6 MongoDB collections
"""

import json
import pandas as pd
import os

---
# 1. Load

In [2]:
INPUT_PATH = "../Data/anilist_anime_data_complete.csv"

print("Loading CSV...")
df = pd.read_csv(INPUT_PATH)
print(f"  Loaded: {len(df)} rows × {len(df.columns)} columns")

Loading CSV...
  Loaded: 20329 rows × 62 columns


---
# 2. Drop Useless columns
### What exactly dropped:

**'Unnamed: 0'** - pandas index<br>
**'type'**       - always "ANIME", no value<br>
**'isFavourite'** 'isLocked' - user-session flags, irrelevant for DB<br>
**'hashtag', 'chapters', 'volumes'** - near 100% null coverImage_large / _medium - redundant<br>
**title_userPreferred** - redundant with title_romaji<br>

In [3]:
DROP_COLS = [
    # pandas / dataset artifacts
    "Unnamed: 0", "type", "isFavourite", "isLocked",
    # manga-only fields
    "hashtag", "chapters", "volumes",
    # redundant image & title variants
    "coverImage_large", "coverImage_medium", "title_userPreferred",
    # redundant with coverImage
    "bannerImage",
    # redundant with averageScore
    "meanScore",
    # alternative titles — 36% null, low priority
    "synonyms",
    # external links — not our platform
    "siteUrl", "externalLinks",
    # low-value metadata
    "seasonInt", "updatedAt", "idMal",
    # AniList-specific editorial — not our platform's ranking
    "rankings",
    # derivable from the DB itself
    "recommendations",
    # near-empty columns (86.2% / 99.9% / 76.1% null)
    "streamingEpisodes", "nextAiringEpisode", "airingSchedule",
]
df.drop(columns=DROP_COLS, inplace=True, errors="ignore")
print(f"  After drop: {len(df.columns)} columns")

  After drop: 39 columns


---
# 3. Fixing Data Types


In [4]:
#Integers
int_cols = [
    "idMal", "startDate_year", "startDate_month", "startDate_day",
    "endDate_year", "endDate_month", "endDate_day",
    "seasonYear", "seasonInt", "episodes", "duration",
    "averageScore", "meanScore", "popularity", "favourites", "trending",
]
for col in int_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

In [5]:
#Bool
bool_cols = ["isLicensed", "isAdult"]
for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].map({True: True, False: False, "True": True, "False": False})

In [6]:
#TimeStamps to datetime
if "updatedAt" in df.columns:
    df["updatedAt"] = pd.to_datetime(df["updatedAt"], unit="s", errors="coerce")

In [7]:
def build_date(row, prefix):
    y = row.get(f"{prefix}_year")
    if pd.isna(y):
        return None
    m = row.get(f"{prefix}_month")
    d = row.get(f"{prefix}_day")
    return {
        "year":  int(y),
        "month": None if pd.isna(m) else int(m),
        "day":   None if pd.isna(d) else int(d),
    }

df["startDate"] = df.apply(lambda r: build_date(r, "startDate"), axis=1)
df["endDate"]   = df.apply(lambda r: build_date(r, "endDate"),   axis=1)

df.drop(columns=[
    "startDate_year","startDate_month","startDate_day",
    "endDate_year","endDate_month","endDate_day",
], inplace=True, errors="ignore")

In [8]:
#Titles
df["title"] = df.apply(lambda r: {
    "romaji":  r.get("title_romaji"),
    "english": r.get("title_english"),
    "native":  r.get("title_native"),
}, axis=1)
df.drop(columns=["title_romaji","title_english","title_native"], inplace=True, errors="ignore")

In [9]:
#CoverImages
df["coverImage"] = df.apply(lambda r: {
    "extraLarge": r.get("coverImage_extraLarge"),
    "color":      r.get("coverImage_color"),
}, axis=1)
df.drop(columns=["coverImage_extraLarge","coverImage_color"], inplace=True, errors="ignore")

In [10]:
#Trailers
df["trailer"] = df.apply(lambda r: {
    "id":        r.get("trailer_id"),
    "site":      r.get("trailer_site"),
    "thumbnail": r.get("trailer_thumbnail"),
} if pd.notna(r.get("trailer_id")) else None, axis=1)
df.drop(columns=["trailer_id","trailer_site","trailer_thumbnail"], inplace=True, errors="ignore")

In [11]:
print("  Data types fixed, nested fields reconstructed")

  Data types fixed, nested fields reconstructed


---
# 4. Basic Data Quality


In [12]:
before = len(df)
df.drop_duplicates(subset=["id"], inplace=True)
print(f"  Duplicates removed: {before - len(df)} rows")

# Drop rows with no meaningful identity (no id or no title)
df = df[df["id"].notna()]
print(f"  Rows after null-id drop: {len(df)}")

for score_col in ["averageScore", "meanScore"]:
    if score_col in df.columns:
        invalid = ((df[score_col] < 0) | (df[score_col] > 100)).sum()
        if invalid > 0:
            print(f"  WARNING: {invalid} out-of-range values in '{score_col}' — clamping")
            df[score_col] = df[score_col].clip(0, 100)

  Duplicates removed: 0 rows
  Rows after null-id drop: 20329


---
# 5. Parsing JSON Columns

In [13]:
JSON_COLS = [
    "genres", "synonyms", "tags", "rankings", "externalLinks",
    "streamingEpisodes", "relations", "characters", "staff", "studios",
    "airingSchedule", "recommendations", "reviews",
    "stats_scoreDistribution", "stats_statusDistribution",
    "nextAiringEpisode",
]

def safe_parse(val):
    if isinstance(val, (list, dict)):
        return val
    if not isinstance(val, str):
        return None
    if val in ("", "[]", "{}"):
        return None
    try:
        return json.loads(val)
    except (json.JSONDecodeError, TypeError):
        return None

for col in JSON_COLS:
    if col in df.columns:
        df[col] = df[col].apply(safe_parse)

print("  JSON columns parsed")

  JSON columns parsed


---
# 6. Final Reports

In [14]:
print("\n=== Final shape ===")
print(f"Rows   : {len(df)}")
print(f"Columns: {len(df.columns)}")
print(f"Columns: {list(df.columns)}")

print("\n=== Null % per column ===")
nulls = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print(nulls[nulls > 0].round(1).to_string())


=== Final shape ===
Rows   : 20329
Columns: 30
Columns: ['id', 'format', 'status', 'description', 'season', 'seasonYear', 'episodes', 'duration', 'countryOfOrigin', 'isLicensed', 'source', 'genres', 'tags', 'averageScore', 'popularity', 'favourites', 'trending', 'isAdult', 'relations', 'characters', 'staff', 'studios', 'reviews', 'stats_scoreDistribution', 'stats_statusDistribution', 'startDate', 'endDate', 'title', 'coverImage', 'trailer']

=== Null % per column ===
reviews         82.3
trailer         63.1
characters      36.1
season          31.3
seasonYear      31.3
relations       30.4
averageScore    20.6
tags            19.2
studios         18.6
staff           13.7
genres          13.1
source          11.4
description      6.1
endDate          1.2
duration         0.9
episodes         0.8
format           0.0


In [15]:
try:
    from pymongo import MongoClient
except ImportError:
    import sys, subprocess as _sp
    _sp.check_call([sys.executable, "-m", "pip", "install", "pymongo"])
    from pymongo import MongoClient

_db = MongoClient("mongodb://localhost:27017")["anilist_db"]
_cols = _db.list_collection_names()

print(f"Collections in 'anilist_db' ({len(_cols)} total):")
for c in _cols:
    print(f"  - {c}: {_db[c].count_documents({})} docs")

Collections in 'anilist_db' (5 total):
  • anime
  • reviews
  • characters
  • tags
  • studios


---
# 7. Extract Collections


In [16]:
SEPARATE_COLS = ["characters", "studios", "tags", "reviews"]

def iter_items(val):
    if isinstance(val, list):
        return val
    if isinstance(val, dict):
        return val.get("nodes") or val.get("edges") or []
    return []

def unwrap_node(item):
    """Flatten GraphQL edge pattern: {isMain, node: {id, name, ...}} → {id, name, ..., isMain}"""
    if not isinstance(item, dict) or "node" not in item:
        return item
    node = item["node"]
    if not isinstance(node, dict):
        return item
    return {**node, **{k: v for k, v in item.items() if k != "node"}}

extracted = {}

for col in SEPARATE_COLS:
    seen, docs, id_lists = set(), [], []

    for val in df[col]:
        items = iter_items(val) if isinstance(val, (list, dict)) else []
        row_ids = []
        for item in items:
            item = unwrap_node(item)
            if not isinstance(item, dict):
                continue
            uid = item.get("id")
            if uid is not None:
                row_ids.append(uid)
                if uid not in seen:
                    seen.add(uid)
                    docs.append(item)
            else:
                docs.append(item)
        id_lists.append(row_ids if row_ids else None)

    extracted[col] = docs
    df[col] = id_lists
    print(f"  {col:15s}: {len(docs):6d} unique docs extracted")

print(f"\n  anime: {len(df)} documents (nested arrays replaced with ID refs)")

  characters     : 144319 unique docs extracted
  studios        :  39745 unique docs extracted
  tags           :    418 unique docs extracted
  reviews        :  13629 unique docs extracted

  anime: 20329 documents (nested arrays replaced with ID refs)


---
# 8. Saving it as JSON
> 6 files → 6 collections in MongoDB

In [17]:
import numpy as np

def convert(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    raise TypeError(f"Object of type {type(obj)} is not JSON serializable")

def sanitize(records):
    return [
        {k: (None if isinstance(v, (float, np.floating)) and not np.isfinite(v) else v)
         for k, v in r.items()}
        for r in records
    ]

# anime collection
anime_records = sanitize(df.where(pd.notna(df), None).to_dict(orient="records"))
anime_path = "../Data/anime.json"
with open(anime_path, "w", encoding="utf-8") as f:
    json.dump(anime_records, f, ensure_ascii=False, default=convert)
print(f"  anime          : {len(anime_records):6d} docs → {anime_path}")

# separate collections
for col, docs in extracted.items():
    out_path = f"../Data/{col}.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(docs, f, ensure_ascii=False, default=convert)
    print(f"  {col:15s}: {len(docs):6d} docs → {out_path}")

print("\n--- mongoimport commands ---")
for col in ["anime"] + list(extracted.keys()):
    print(f"mongoimport --db anilist_db --collection {col} --file Data/{col}.json --jsonArray")

  anime          :  20329 docs → ../Data/anime.json
  characters     : 144319 docs → ../Data/characters.json
  studios        :  39745 docs → ../Data/studios.json
  tags           :    418 docs → ../Data/tags.json
  reviews        :  13629 docs → ../Data/reviews.json

--- mongoimport commands ---
mongoimport --db anilist_db --collection anime --file Data/anime.json --jsonArray
mongoimport --db anilist_db --collection characters --file Data/characters.json --jsonArray
mongoimport --db anilist_db --collection studios --file Data/studios.json --jsonArray
mongoimport --db anilist_db --collection tags --file Data/tags.json --jsonArray
mongoimport --db anilist_db --collection reviews --file Data/reviews.json --jsonArray


In [18]:
try:
    from pymongo import MongoClient
    import bson
except ImportError:
    import sys, subprocess as _sp
    _sp.check_call([sys.executable, "-m", "pip", "install", "pymongo"])
    from pymongo import MongoClient
    import bson

MONGO_HOST = "mongodb://localhost:27017"
MONGO_DB   = "anilist_db"
MAX_BSON   = 15 * 1024 * 1024  # 1 MB safety margin under MongoDB's 16 MB hard limit
BATCH      = 1000

client = MongoClient(MONGO_HOST)
db = client[MONGO_DB]

def doc_size(d):
    return len(bson.BSON.encode(d))

def trim_oversize(d):
    """Cut down nested arrays/strings until the doc is under 15 MB."""
    for field in ("staff", "relations"):
        if doc_size(d) <= MAX_BSON:
            return d
        v = d.get(field)
        if isinstance(v, list) and len(v) > 50:
            d[field] = v[:50]
    if doc_size(d) > MAX_BSON and isinstance(d.get("description"), str):
        d["description"] = d["description"][:5000]
    return d

collections = ["anime"] + list(extracted.keys())

print(f"Importing into {MONGO_DB} via PyMongo...\n")
for col in collections:
    with open(f"../Data/{col}.json", encoding="utf-8") as f:
        records = json.load(f)

    trimmed = 0
    if col == "anime":
        for d in records:
            if doc_size(d) > MAX_BSON:
                trim_oversize(d)
                trimmed += 1

    db[col].drop()

    inserted = 0
    for i in range(0, len(records), BATCH):
        chunk = records[i:i + BATCH]
        try:
            inserted += len(db[col].insert_many(chunk, ordered=False).inserted_ids)
        except Exception as e:
            inserted += getattr(e, "details", {}).get("nInserted", 0)
            print(f"  WARN  {col} batch@{i}: {type(e).__name__} {str(e)[:80]}")

    final = db[col].count_documents({})
    flag = "OK" if final == len(records) else "PARTIAL"
    extra = f" (trimmed {trimmed} oversize docs)" if trimmed else ""
    print(f"  [{flag}] {col:11s} {inserted}/{len(records)} inserted, count={final}{extra}")

print("\nDone. Connect: mongodb://localhost:27017")

Importing into MongoDB...


FileNotFoundError: [WinError 2] 系统找不到指定的文件。

In [ ]:
all_collections = {"anime": df, **{col: pd.DataFrame(docs) for col, docs in extracted.items()}}

for name, data in all_collections.items():
    frame = data if isinstance(data, pd.DataFrame) else pd.DataFrame(data)
    print(f"\n{'='*50}")
    print(f"  {name.upper()}")
    print(f"{'='*50}")
    print(f"  Rows   : {len(frame)}")
    print(f"  Columns: {len(frame.columns)}")
    print(f"  Fields : {list(frame.columns)}")

    nulls = (frame.isnull().sum() / len(frame) * 100).sort_values(ascending=False)
    non_zero = nulls[nulls > 0].round(1)
    if non_zero.empty:
        print("  Null % : none")
    else:
        print("  Null % per column:")
        print(non_zero.to_string())

---
# 9. Collection Samples
> First 10 documents per collection — for visual orientation.

In [ ]:
from IPython.display import display

all_collections = {"anime": df, **{col: pd.DataFrame(docs) for col, docs in extracted.items()}}

for name, data in all_collections.items():
    frame = data if isinstance(data, pd.DataFrame) else pd.DataFrame(data)
    print(f"\n{'='*60}")
    print(f"  {name.upper()} — {len(frame)} total docs")
    print(f"{'='*60}")
    display(frame.head(10))